# Chapter 9: Tool calling & MCP

We give an LLM four tools over ShopFlow (look up an order, find orders by email, check stock,
propose a refund), build the agent loop with the plain OpenAI SDK, then serve the **same** tools
over MCP (stdio and streamable HTTP) and run the same agent through an MCP client.

Run from this folder: `cd ch09/01_main-chapter-code && uv run jupyter lab`. Prerequisites:
ShopFlow DB running and seeded (`datasets/shopflow`), `.env` filled in, Ollama running for the
local-model rows.

In [ ]:
import json, socket, statistics, subprocess, sys, time
from pathlib import Path

CODE = Path.cwd()
sys.path.insert(0, str(CODE))

import ch09_agent, ch09_mcp_client, ch09_refunds, ch09_setup, ch09_tools
import ch09_scenarios as sc
from llmprod import chat_client, connect, get_settings

ch09_setup.setup_tables()   # idempotent: creates ch09_pending_refunds if missing
sc.reset_all()              # start from an empty refund queue
settings = get_settings()
MODEL = settings.llm_model
print("model:", MODEL, "|", settings.llm_base_url)
with connect() as conn:
    print("orders in DB:", conn.execute("select count(*) from orders").fetchone()[0])

## 1. What a tool call actually is

The model never runs anything. We describe each tool as JSON Schema, the model answers with a
**request** to call one (`tool_calls`), *we* run it, and we send the result back as a `tool`
message. Here is one round trip by hand, before any loop.

In [ ]:
toolbox = ch09_tools.LocalToolbox()
print(json.dumps(toolbox.specs[0], indent=2))

In [ ]:
client = chat_client()
ctx0 = sc.pick_context(offset=0)
oid = ctx0["small"]["id"]
messages = [
    {"role": "system", "content": ch09_agent.SYSTEM_PROMPT},
    {"role": "user", "content": f"What's the status of order {oid}?"},
]
r1 = client.chat.completions.create(
    model=MODEL, messages=messages, tools=toolbox.specs, max_tokens=512, reasoning_effort="none"
)
choice = r1.choices[0]
print("finish_reason:", choice.finish_reason)
call = choice.message.tool_calls[0]
print("tool call    :", call.function.name, call.function.arguments)
print("usage        :", r1.usage.prompt_tokens, "prompt +", r1.usage.completion_tokens, "completion tokens")

In [ ]:
result = toolbox.call(call.function.name, call.function.arguments)   # WE run the tool
print(result[:300], "...")
messages += [
    {"role": "assistant", "content": "", "tool_calls": [
        {"id": call.id, "type": "function",
         "function": {"name": call.function.name, "arguments": call.function.arguments}}]},
    {"role": "tool", "tool_call_id": call.id, "content": result},
]
r2 = client.chat.completions.create(
    model=MODEL, messages=messages, tools=toolbox.specs, max_tokens=512, reasoning_effort="none"
)
print("\nfinal answer:", r2.choices[0].message.content)

## 2. v0: the naive loop

The smallest loop that works: call the model, run whatever it asks, repeat until it stops asking.
It calls the Python functions directly and trusts the model's arguments.

In [ ]:
def naive_agent(question):
    messages = [{"role": "system", "content": ch09_agent.SYSTEM_V0},
                {"role": "user", "content": question}]
    while True:                                           # no step limit
        msg = client.chat.completions.create(
            model=MODEL, messages=messages, tools=toolbox.specs, max_tokens=512,
            reasoning_effort="none").choices[0].message
        if not msg.tool_calls:
            return msg.content
        messages.append({"role": "assistant", "content": msg.content or "", "tool_calls": [
            {"id": t.id, "type": "function",
             "function": {"name": t.function.name, "arguments": t.function.arguments}}
            for t in msg.tool_calls]})
        for t in msg.tool_calls:
            fn = ch09_tools.TOOL_FUNCTIONS[t.function.name]      # trusts the name
            out = fn(**json.loads(t.function.arguments))         # trusts the arguments
            messages.append({"role": "tool", "tool_call_id": t.id, "content": json.dumps(out, default=str)})

print(naive_agent(f"What's the status of order {oid}?"))

In [ ]:
try:
    print(naive_agent("What's the status of order 99999999?"))
except Exception as exc:
    print(type(exc).__name__, "->", exc)

## 3. Measuring v0: what the hardened loop changes

The first user who mistypes an order id crashes v0. The fix is not a try/except in the loop; it is
a rule about *who the error is for*. A tool failure is information for the model, so it becomes a
`tool` message. `LocalToolbox.call` never raises. Each row below is a different way for a call to go
wrong, with the exact message the model receives.

In [ ]:
bad_calls = [
    ("nonexistent order", "get_order", '{"order_id": 99999999}'),
    ("unknown tool", "delete_everything", "{}"),
    ("truncated JSON", "get_order", '{"order_id": 12'),
    ("wrong type", "get_order", '{"order_id": "twelve"}'),
    ("invented argument", "check_inventory", '{"sku": "SF-ELE-0001", "warehouse": "TX-01"}'),
    ("dollars instead of cents (negative)", "create_refund",
     '{"order_id": 2, "amount_cents": -25, "reason": "damaged", "idempotency_key": "k"}'),
    ("reason not in the enum", "create_refund",
     '{"order_id": 2, "amount_cents": 2500, "reason": "item broke", "idempotency_key": "k"}'),
    ("malformed email", "search_orders", '{"customer_email": "olivia"}'),
]
for label, name, raw in bad_calls:
    print(f"{label:38s} -> {toolbox.call(name, raw)}")

## 4. Does tool calling work on each provider?

OpenAI's model page for `gpt-6-luna` says Chat Completions supports function calling *only* with
`reasoning_effort` set to `none`. We test that through OpenRouter, with and without the parameter,
and on a second hosted model and a local one. The same question, three runs each (two for the
local model).

In [ ]:
import openai

ollama = openai.OpenAI(base_url=settings.embed_base_url, api_key="ollama", timeout=300)
oid, sku = ctx0["small"]["id"], ctx0["stock"]["sku"]
Q = f"Is order {oid} delivered, and how many units of {sku} are in stock in total?"
want_total = int(ctx0["stock"]["total"])

CONFIGS = [
    ("openai/gpt-6-luna", "no reasoning_effort", None, client, 3),
    ("openai/gpt-6-luna", 'reasoning_effort="none"', {"reasoning_effort": "none"}, client, 3),
    ("openai/gpt-6-luna", 'reasoning_effort="low"', {"reasoning_effort": "low"}, client, 3),
    ("anthropic/claude-haiku-5.5", "no reasoning_effort", None, client, 3),
    ("anthropic/claude-haiku-5.5", 'reasoning_effort="none"', {"reasoning_effort": "none"}, client, 3),
    ("qwen3:8b (Ollama)", 'reasoning_effort="none"', {"reasoning_effort": "none"}, ollama, 2),
]
compat = []
for model, label, extra, cl, reps in CONFIGS:
    for _ in range(reps):
        name = "qwen3:8b" if model.startswith("qwen3") else model
        res = ch09_agent.run_agent(Q, toolbox, client=cl, model=name, extra=extra, max_tokens=1024)
        used = {c.name for c in res.tool_calls}
        ok = {"get_order", "check_inventory"} <= used and str(want_total) in res.answer.replace(",", "")
        compat.append(dict(model=model, setting=label, ok=ok, stop=res.stop_reason, steps=res.steps,
                           parallel=res.max_parallel, tokens=res.total_tokens, cost=res.cost_usd,
                           ms=res.latency_ms, note="" if res.stop_reason == "done" else res.answer[:160]))
import pandas as pd
cdf = pd.DataFrame(compat)
summary = (cdf.groupby(["model", "setting"], sort=False)
    .agg(ok=("ok", "sum"), runs=("ok", "size"), max_parallel=("parallel", "max"),
         tokens=("tokens", "mean"), cost_usd=("cost", "mean"), p50_ms=("ms", "median")))
summary["tokens"] = summary["tokens"].round(0).astype(int)
summary["cost_usd"] = summary["cost_usd"].round(5)
summary

In [ ]:
print(cdf[cdf.note != ""][["model", "setting", "stop", "note"]].to_string())

## 5. The scenario suite

Fourteen scenarios (listed in `ch09_scenarios.py`), each with a mechanical check on tool calls and
database state, never on how the answer reads. Five configurations run concurrently, each on its own
set of orders: `gpt-6-luna` and `claude-haiku-5.5` (3 repetitions each), `qwen3:8b` on Ollama (1),
and `gpt-6-luna` through the **MCP server** over stdio (1) and over streamable HTTP (1).
We start the HTTP server here and stop it at the end of the cell.

In [ ]:
HTTP_URL = "http://127.0.0.1:8009/mcp"

def port_open(port):
    with socket.socket() as s:
        s.settimeout(0.5)
        return s.connect_ex(("127.0.0.1", port)) == 0

assert not port_open(8009), "port 8009 is already in use"
log = open("mcp_http_server.log", "w")
server = subprocess.Popen(
    ["uv", "run", "python", "ch09_mcp_server.py", "--transport", "http", "--port", "8009"],
    cwd=CODE, stdout=log, stderr=subprocess.STDOUT)
for _ in range(120):
    if port_open(8009):
        break
    time.sleep(0.5)
print("HTTP MCP server listening on :8009 ->", port_open(8009))

In [ ]:
from concurrent.futures import ThreadPoolExecutor

NONE = {"reasoning_effort": "none"}
suites = {}
try:
    with ch09_mcp_client.McpToolbox(ch09_mcp_client.stdio_server()) as mcp_stdio, \
         ch09_mcp_client.McpToolbox(HTTP_URL) as mcp_http:
        print("MCP protocol negotiated: stdio =", mcp_stdio.protocol_version, "| http =", mcp_http.protocol_version)
        jobs = {
            "luna / local": ("openai/gpt-6-luna", toolbox, 0, 3, NONE, None),
            "haiku / local": ("anthropic/claude-haiku-5.5", toolbox, 1, 3, None, None),
            "qwen3:8b / local": ("qwen3:8b", toolbox, 2, 1, NONE, ollama),
            "luna / MCP stdio": ("openai/gpt-6-luna", mcp_stdio, 3, 1, NONE, None),
            "luna / MCP http": ("openai/gpt-6-luna", mcp_http, 4, 1, NONE, None),
        }
        t0 = time.perf_counter()
        with ThreadPoolExecutor(max_workers=len(jobs)) as pool:
            futs = {label: pool.submit(sc.run_suite, label, model, tb, offset=off, reps=reps,
                                       extra=extra, client=cl)
                    for label, (model, tb, off, reps, extra, cl) in jobs.items()}
            suites = {label: f.result() for label, f in futs.items()}
        print(f"all suites finished in {time.perf_counter() - t0:.0f}s")
finally:
    server.terminate()
    server.wait(timeout=15)
    log.close()
print("HTTP server stopped; port 8009 open:", port_open(8009))

In [ ]:
summ = pd.DataFrame([sc.summarize(s) for s in suites.values()]).set_index("label")
summ["pass_rate"] = (100 * summ["pass"] / summ["total"]).round(1)
summ["cost_usd"] = summ["cost_usd"].round(4)
summ[["pass", "total", "pass_rate", "tool_errors", "tokens_per_run", "p50_ms", "p95_ms", "cost_usd"]]

In [ ]:
ids = [s.id for s in sc.SCENARIOS]
matrix = {}
for label, s in suites.items():
    col = {}
    for i in ids:
        runs = [r for r in s["runs"] if r["id"] == i]
        col[i] = f"{sum(r['passed'] for r in runs)}/{len(runs)}"
    matrix[label] = col
pd.DataFrame(matrix)

In [ ]:
fails = [(label, r["rep"], r["id"], r["detail"], r["tool_calls"], r["answer"][:200].replace("\n", " "))
         for label, s in suites.items() for r in s["runs"] if not r["passed"]]
print(len(fails), "failing runs")
for f in fails:
    print(f)

In [ ]:
rows = []
for label, s in suites.items():
    for r in s["runs"]:
        rows.append({"label": label, **{k: r[k] for k in ("id", "passed", "steps", "max_parallel", "tool_errors", "tokens", "latency_ms")}})
runs_df = pd.DataFrame(rows)
print("runs that issued >=2 tool calls in one assistant turn:",
      int((runs_df.max_parallel >= 2).sum()), "of", len(runs_df))
print("tool errors seen by the model (all suites):", int(runs_df.tool_errors.sum()))
runs_df[runs_df.tool_errors > 0].groupby(["label", "id"]).tool_errors.sum()

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
colors = ["#4a7fd1", "#8a5cc9", "#c9961c", "#3f9a5b", "#3f9a5b"]
ax[0].bar(summ.index, summ["pass_rate"], color=colors)
ax[0].set_ylim(0, 105); ax[0].set_ylabel("scenario pass rate (%)")
ax[0].set_title("14 scenarios, mechanical checks")
for i, v in enumerate(summ["pass_rate"]):
    ax[0].text(i, v + 1.5, f"{v:.0f}%", ha="center", fontsize=9)
ax[1].bar(summ.index, summ["p50_ms"] / 1000, color=colors)
ax[1].set_ylabel("p50 seconds per conversation"); ax[1].set_title("latency (median)")
for a in ax:
    a.tick_params(axis="x", labelrotation=25)
plt.tight_layout()
Path("../images").mkdir(exist_ok=True)
plt.savefig("../images/09-scenarios.png", dpi=130)
print("saved ../images/09-scenarios.png")

## 6. Hard stops with a live model

`max_steps` and `token_budget` exist for the day a model loops or a tool returns a novel. We trigger
both on purpose with a real model and a question that needs several steps.

In [ ]:
q = (f"Compare orders {ctx0['pair']['a_id']} and {ctx0['pair']['b_id']}, then check the stock of "
     f"{ctx0['single']['sku']} and tell me which order has more items.")
for kwargs in ({"max_steps": 8, "token_budget": 20_000}, {"max_steps": 1}, {"token_budget": 1_500}):
    r = ch09_agent.run_agent(q, toolbox, client=client, model=MODEL, extra=NONE, **kwargs)
    print(kwargs, "->", r.stop_reason, "| steps", r.steps, "| tool calls run", len(r.tool_calls),
          "| tokens", r.total_tokens)
    print("   answer:", r.answer[:150].replace("\n", " "))
    if r.stop_reason != "done":
        print("   last message:", r.messages[-1]["content"])

## 7. The human side: `approve.py`

The model can only *propose*. We have the agent queue two refunds (one within the agent limit, one
over it), then act as different humans with the CLI.

In [ ]:
ctx7 = sc.pick_context(offset=7)
sc.clear_refunds([ctx7["small"]["id"], ctx7["mid"]["id"]])
for s_ in (sc.SCENARIOS[5], sc.SCENARIOS[6]):            # S6 within limit, S7 over limit
    r = ch09_agent.run_agent(s_.question(ctx7), toolbox, client=client, model=MODEL, extra=NONE)
    print(s_.id, "->", r.answer[:170].replace("\n", " "))

def cli(*args):
    p = subprocess.run(["uv", "run", "python", "approve.py", *args], cwd=CODE, capture_output=True, text=True)
    print("$ approve.py", " ".join(args), f"[exit {p.returncode}]")
    print((p.stdout + p.stderr).rstrip())

cli("list")

In [ ]:
rows = [r for r in ch09_refunds.list_requests("pending") if r["order_id"] in (ctx7["small"]["id"], ctx7["mid"]["id"])]
lead_id = next(r["id"] for r in rows if r["required_approver"] == "team_lead")
agent_id = next(r["id"] for r in rows if r["required_approver"] == "agent")
cli("approve", str(lead_id), "--role", "agent")                       # too junior: refused
cli("approve", str(lead_id), "--role", "team_lead", "--note", "defect confirmed")
cli("reject", str(agent_id), "--role", "agent", "--note", "outside return window")
cli("approve", str(lead_id), "--role", "finance")                    # already decided
cli("list", "--status", "all")

## 8. MCP: the same tools, a different door

`ch09_mcp_server.py` registers the very functions in `ch09_tools.py`. We check what a client sees
over stdio: protocol version, tool list, the safety hints, and that the converted schemas match the
ones the raw loop used.

In [ ]:
from mcp.client.client import Client

async with Client(ch09_mcp_client.stdio_server()) as mc:
    print("server :", mc.server_info.name, mc.server_info.version, "| protocol", mc.protocol_version)
    tools = (await mc.list_tools()).tools
    for t in tools:
        a = t.annotations
        print(f"  {t.name:16s} read_only={a.read_only_hint!s:5} idempotent={a.idempotent_hint}")
    converted = [ch09_mcp_client.mcp_tool_to_openai(t) for t in tools]

local = {s["function"]["name"]: s["function"]["parameters"] for s in toolbox.specs}
for c in converted:
    n = c["function"]["name"]
    same = (set(c["function"]["parameters"]["properties"]) == set(local[n]["properties"])
            and set(c["function"]["parameters"].get("required", [])) == set(local[n].get("required", [])))
    print(f"schema parity for {n}: {same}")
print(json.dumps(converted[3]["function"]["parameters"]["properties"]["reason"]))

In [ ]:
with ch09_mcp_client.McpToolbox(ch09_mcp_client.stdio_server()) as tb:
    print(tb.call("get_order", '{"order_id": 99999999}'))
    print(tb.call("create_refund", '{"order_id": 2, "amount_cents": 0, "reason": "late", "idempotency_key": "k"}')[:230])

In [ ]:
sc.reset_all()
out = Path("results"); out.mkdir(exist_ok=True)
(out / "scenario_suites.json").write_text(json.dumps(
    {"compat": compat, "suites": suites, "summary": json.loads(summ.reset_index().to_json(orient="records"))},
    indent=1, default=str))
print("saved results/scenario_suites.json")